<a href="https://colab.research.google.com/github/kirti73/AI_lab/blob/main/1BF24CS140_WEEK6_PROPOSTIONAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class Formula:
    """Base class for propositional logic formulas."""
    def evaluate(self, model):
        raise NotImplementedError()

    def symbols(self):
        raise NotImplementedError()

class Symbol(Formula):
    def __init__(self, name):
        self.name = name

    def evaluate(self, model):
        return model.get(self.name, False)

    def symbols(self):
        return {self.name}

    def __str__(self):
        return self.name

class And(Formula):
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def evaluate(self, model):
        return self.left.evaluate(model) and self.right.evaluate(model)

    def symbols(self):
        return self.left.symbols().union(self.right.symbols())

    def __str__(self):
        return f"({self.left} & {self.right})"

class Or(Formula):
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def evaluate(self, model):
        return self.left.evaluate(model) or self.right.evaluate(model)

    def symbols(self):
        return self.left.symbols().union(self.right.symbols())

    def __str__(self):
        return f"({self.left} | {self.right})"

class Implies(Formula):
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def evaluate(self, model):
        return (not self.left.evaluate(model)) or self.right.evaluate(model)

    def symbols(self):
        return self.left.symbols().union(self.right.symbols())

    def __str__(self):
        return f"({self.left} => {self.right})"

class Not(Formula):
    def __init__(self, formula):
        self.formula = formula

    def evaluate(self, model):
        return not self.formula.evaluate(model)

    def symbols(self):
        return self.formula.symbols()

    def __str__(self):
        return f"~{self.formula}"


def tt_entails(kb, alpha):
    """Returns True if the knowledge base kb entails the query alpha, and False otherwise."""
    symbols = list(kb.symbols().union(alpha.symbols()))
    return tt_check_all(kb, alpha, symbols, {})


def tt_check_all(kb, alpha, symbols, model):
    """Helper function for truth-table enumeration checking."""
    if not symbols:
        if kb.evaluate(model):
            return alpha.evaluate(model)
        else:
            return True  # when KB is false, always return true
    else:
        P = symbols[0]
        rest = symbols[1:]

        # Try setting P to True
        model_true = model.copy()
        model_true[P] = True

        # Try setting P to False
        model_false = model.copy()
        model_false[P] = False

        return (tt_check_all(kb, alpha, rest, model_true) and
                tt_check_all(kb, alpha, rest, model_false))

# --- Quick Demo ---
# Let's verify Modus Ponens:
# KB: A and (A => B)
# Query: B
A = Symbol('A')
B = Symbol('B')

kb = And(A, Implies(A, B))
query = B

print(f"Knowledge Base: {kb}")
print(f"Query: {query}")
entails = tt_entails(kb, query)
print(f"Does KB entail Query? {entails}")

Knowledge Base: (A & (A => B))
Query: B
Does KB entail Query? True


In [ ]:
R = Symbol('R')
S = Symbol('S')

# (i) a: (R -> S) & (S -> R)
a = And(Implies(R, S), Implies(S, R))

# (ii) b: R <-> S
# (R <-> S) is logically equivalent to (R -> S) & (S -> R)
b = And(Implies(R, S), Implies(S, R))

# (iii) c: ~R | S
c = Or(Not(R), S)

# (iv) d: ~(R <-> S)
# Since <-> is represented as (R -> S) & (S -> R), its negation is:
d = Not(And(Implies(R, S), Implies(S, R)))

# (v) e: (R & S) | (~R & ~S)
e = Or(And(R, S), And(Not(R), Not(S)))

# Perform the requested checks
entails_a_b = tt_entails(a, b)
entails_a_e = tt_entails(a, e)

print(f"Relation a: {a}")
print(f"Relation b: {b}")
print(f"Relation e: {e}\n")
print(f"(i) Does 'a' entail 'b'? {entails_a_b}")
print(f"(ii) Does 'a' entail 'e'? {entails_a_e}")

Relation a: ((R => S) & (S => R))
Relation b: ((R => S) & (S => R))
Relation e: ((R & S) | (~R & ~S))

(i) Does 'a' entail 'b'? True
(ii) Does 'a' entail 'e'? True
